# scDORI data preprocessing

This notebook prepares paired RNA and ATAC data for scDORI. We select genes, transcription factors (TFs) and accessible DNA regions, then estimate TF binding from motifs and the data.

You will need **AnnData files containing raw RNA and ATAC counts**, with matching cell identifiers and a sample label for each cell. Cell calling and quality control should already be complete; keep the unnormalised counts in `.X`. Use gene symbols for RNA and peak names such as `chr1:1000-1200` for ATAC. The peaks and genome reference must use the same genome assembly. Later, downstream analysis uses a column called `celltype` in the RNA cell annotations; cluster labels can also be used there.

Copy `configs/scdori.example.json` to `scdori_config.json` in the repository folder. In its `preprocessing` section, enter your data paths, species, genome and motif database. Set `preprocessing.batch_key` and `training.batch_col` to the same sample-label column in your two input files. All three notebooks use this same file. Keep it if you have already set up your project.

Once scDORI and BEDTools are installed, run the cells in order. The [setup guide](../installation.md#install-and-configure) covers installation. The saved outputs below show the single-cell gastrulation example, which took about 13 minutes on 64 cores; grouping cells into metacells (6 minutes) and the distance table (3 minutes) took longest.


In [1]:
import logging
from pathlib import Path
import numpy as np
import pandas as pd

from scdori.pp import (
    compute_gene_peak_distance_matrix,
    compute_hvgs_and_tfs,
    compute_in_silico_chipseq,
    compute_motif_scores,
    create_dir_if_not_exists,
    create_extended_gene_bed,
    create_metacells,
    download_genome_references,
    filter_protein_coding_genes,
    intersect_cells,
    keep_promoters_and_select_hv_peaks,
    retain_requested_peaks,
    load_anndata,
    load_gtf,
    load_motif_database,
    metacell_correlation_matrices,
    remove_mitochondrial_genes,
    run_bedtools_intersect,
    save_processed_datasets,
)

from scdori.configuration import load_notebook_config, check_notebook_inputs, save_notebook_config

# Optional: set a path here. Otherwise find scdori_config.json in this directory
# or a parent; without JSON, the existing Python config.py settings still work.
CONFIG_FILE = None
settings = load_notebook_config(CONFIG_FILE)
ppConfig = settings.preprocessing

logger = logging.getLogger(__name__)

Configuration: scdori_config.json; path root: .


In [2]:
check_notebook_inputs(settings, "preprocessing")
logging.getLogger().setLevel(ppConfig.logging_level)

logger.info("=== Starting multi-ome preprocessing pipeline ===")

preprocessing: 3 required input files found


## 1. Prepare directories

In [3]:
data_dir = Path(ppConfig.data_dir)
genome_dir = Path(ppConfig.genome_dir)
motif_dir = Path(ppConfig.motif_directory)
out_dir = data_dir / ppConfig.output_subdir_name

create_dir_if_not_exists(genome_dir)
create_dir_if_not_exists(motif_dir)
create_dir_if_not_exists(out_dir)

INFO	Creating directory: data/generated


## 2. Download reference genome, gene annotations and chromosome sizes

In [4]:
download_genome_references(
    genome_dir=genome_dir,
    species=ppConfig.species,
    assembly=ppConfig.genome_assembly,
    gtf_url=ppConfig.gtf_url,
    chrom_sizes_url=ppConfig.chrom_sizes_url,
    fasta_url=ppConfig.fasta_url,
)

INFO	Using genome references for species='mouse', assembly='mm10'.
GTF: https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_mouse/release_M18/gencode.vM18.basic.annotation.gtf.gz
Chrom.sizes: https://hgdownload.soe.ucsc.edu/goldenPath/mm10/bigZips/mm10.chrom.sizes
FASTA: https://hgdownload.soe.ucsc.edu/goldenPath/mm10/bigZips/mm10.fa.gz


INFO	Reference files are ready in references/genome


## 3. Load the RNA and ATAC data

In [5]:
data_rna, data_atac = load_anndata(
    data_dir, ppConfig.rna_adata_file_name, ppConfig.atac_adata_file_name
)

INFO	Loading RNA from data/anndata.h5ad, ATAC from data/PeakMatrix_anndata.h5ad


## 4. Match the cells and remove mitochondrial genes

In [6]:
data_rna, data_atac = intersect_cells(data_rna, data_atac)
data_rna = remove_mitochondrial_genes(
    data_rna, mito_prefix=ppConfig.mitochondrial_prefix
)

INFO	Intersected cells: now RNA=(56861, 32285), ATAC=(56861, 192251)


INFO	Removed 13 mitochondrial genes with prefix=mt-


## 5. Keep protein-coding genes

We use the reference annotation to identify protein-coding genes in the RNA data. These genes are carried forward to feature selection.


In [7]:
gtf_file = genome_dir / "annotation.gtf"
gtf_df = load_gtf(gtf_file)
data_rna = filter_protein_coding_genes(data_rna, gtf_df)

INFO	Loading GTF from references/genome/annotation.gtf


INFO	Filtered to protein-coding genes: 19374 genes left.


## 6. Select genes and TFs

We select variable genes and TFs with motifs in the supplied database. To include genes or TFs of particular interest, add their names to `genes_user` or `tfs_user` in the `preprocessing` section of `scdori_config.json`. For example:

```json
"genes_user": ["Gypa", "Alas2"],
"tfs_user": ["Tal1", "Gata1"]
```

Use names present in your filtered RNA data. TFs also need a matching motif. Put TFs you want networks for in `tfs_user`, even if you also include them as genes.

`num_genes` is the target number of RNA features, including the `num_tfs` TFs. The remaining features are chosen by variability. Requested features take priority, so the final number may be larger. The report below lists any requested names that could not be included.


In [8]:
motif_path = motif_dir / f"{ppConfig.motif_database}_{ppConfig.species}.meme"
tf_names_all = []
with open(motif_path) as f:
    for line in f:
        if line.startswith("MOTIF"):
            parts = line.strip().split()
            if len(parts) >= 3:
                tf_name = parts[2].split("_")[0].strip("()").strip()
                tf_names_all.append(tf_name)
tf_names_all = sorted(list(set(tf_names_all)))

data_rna, final_genes, final_tfs = compute_hvgs_and_tfs(
    data_rna=data_rna,
    tf_names=tf_names_all,
    user_genes=ppConfig.genes_user,
    user_tfs=ppConfig.tfs_user,
    num_genes=ppConfig.num_genes,
    num_tfs=ppConfig.num_tfs,
    min_cells=ppConfig.min_cells_per_gene,
)

import json
print(json.dumps(data_rna.uns["scdori_requested_features"], indent=2))

INFO	Selecting HVGs and TFs...


INFO	Selected 3700 HVGs + 300 TFs.


{
  "genes_retained": [],
  "tfs_retained": [],
  "genes_missing_after_filtering": [],
  "tfs_missing_after_filtering": [],
  "tfs_without_motifs": []
}


## 7. Define the genomic window around each gene

We extend each gene's interval by `window_size` on each side. By default, this is 150 kb for human and 80 kb for mouse. This defines which nearby peaks scDORI will consider when learning peak–gene relationships.


In [9]:
chrom_sizes_path = genome_dir / f"{ppConfig.genome_assembly}.chrom.sizes"
extended_genes_bed_df = create_extended_gene_bed(
    gtf_df,
    final_genes + final_tfs,  # if we want to include TF genes too
    window_size=ppConfig.window_size,
    chrom_sizes_path=chrom_sizes_path,
)

gene_bed_file = out_dir / f"genes_extended_{ppConfig.window_size//1000}kb.bed"
extended_genes_bed_df.to_csv(gene_bed_file, sep="\t", header=False, index=False)
logger.info(f"Created extended gene bed => {gene_bed_file}")

In [10]:
data_atac.var

,idx,chr,start,end
chr1:3035602-3036202,chr1:3035602-3036202,chr1,3035602,3036202
chr1:3062653-3063253,chr1:3062653-3063253,chr1,3062653,3063253
chr1:3072313-3072913,chr1:3072313-3072913,chr1,3072313,3072913
chr1:3191496-3192096,chr1:3191496-3192096,chr1,3191496,3192096
chr1:3340575-3341175,chr1:3340575-3341175,chr1,3340575,3341175
...,...,...,...,...
chrX:169902806-169903406,chrX:169902806-169903406,chrX,169902806,169903406
chrX:169905921-169906521,chrX:169905921-169906521,chrX,169905921,169906521
chrX:169915616-169916216,chrX:169915616-169916216,chrX,169915616,169916216
chrX:169925487-169926087,chrX:169925487-169926087,chrX,169925487,169926087


## 8. Prepare peak coordinates

We make a table of chromosome, start and end positions from the ATAC peak names. These coordinates will be used to find nearby genes and extract DNA sequences.


In [11]:
data_atac.var["chr"] = [x.split(":")[0] for x in data_atac.var.index]
data_atac.var["start"] = [
    int(x.split(":")[1].split("-")[0]) for x in data_atac.var.index
]
data_atac.var["end"] = [int(x.split(":")[1].split("-")[1]) for x in data_atac.var.index]
data_atac.var["peak_name"] = data_atac.var.index
all_peaks_bed = out_dir / "peaks_all.bed"
data_atac.var[["chr", "start", "end", "peak_name"]].to_csv(
    all_peaks_bed, sep="\t", header=False, index=False
)

## 9. Find peaks near the selected genes

We keep peaks within the gene windows. You can also request peaks by adding their exact ATAC feature names to `preprocessing.peaks_user` in your config.

Requested peaks are kept even if they fall outside these windows, so they can contribute to the ATAC model. The distance settings still determine which genes they can be linked to. Missing names are listed in the request report.


In [12]:
#
intersected_bed = out_dir / "peaks_intersected.bed"
run_bedtools_intersect(
    a_bed=all_peaks_bed, b_bed=gene_bed_file, out_bed=intersected_bed
)

peaks_intersected = (pd.read_csv(intersected_bed, sep="\t", header=None)
                    if intersected_bed.stat().st_size else pd.DataFrame(columns=range(4)))
peaks_intersected.columns = ["chr", "start", "end", "peak_name"]
windowed_set = set(peaks_intersected["peak_name"])

# Subset data_atac to these peaks
data_atac = retain_requested_peaks(data_atac, windowed_set, ppConfig.peaks_user)
logger.info(f"After gene-window filtering => shape={data_atac.shape}")

INFO	Running: ['bedtools', 'intersect', '-u', '-wa', '-a', 'data/generated/peaks_all.bed', '-b', 'data/generated/genes_extended_80kb.bed']


## 10. Group similar cells for preprocessing

We group cells with similar RNA profiles into small groups called *metacells*. Averaging counts within each group helps estimate variable peaks and TF-binding scores from sparse data. The saved training inputs still contain the original single cells.


In [13]:
#
rna_metacell, atac_metacell = create_metacells(
    data_rna,
    data_atac,
    grouping_key="leiden",
    resolution=ppConfig.leiden_resolution,
    batch_key=ppConfig.batch_key,
)
# Copy labels
data_atac.obs["leiden"] = data_rna.obs["leiden"]

INFO	Creating metacells with resolution=10 (grouping key=leiden).


2026-11-03 03:21:16,977 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


2026-11-03 03:21:19,681 - harmonypy - INFO - sklearn.KMeans initialization complete.


2026-11-03 03:21:19,842 - harmonypy - INFO - Iteration 1 of 10


2026-11-03 03:21:29,562 - harmonypy - INFO - Iteration 2 of 10


2026-11-03 03:21:39,490 - harmonypy - INFO - Iteration 3 of 10


... (2 outputs left out)

2026-11-03 03:22:08,828 - harmonypy - INFO - Iteration 6 of 10


2026-11-03 03:22:18,581 - harmonypy - INFO - Converged after 6 iterations


src/scdori/pp/metacells.py:83: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(working, resolution=resolution, key_added=grouping_key)


INFO	Metacell shapes: RNA=(134, 4000), ATAC=(134, 98556)


## 11. Select peaks

We keep your requested peaks and select variable peaks up to `num_peaks`. Requested peaks can make the final number larger than this target.

Promoter peaks can also be kept when `keep_promoter_peaks=true` and the ATAC feature table contains a Boolean promoter annotation. Set `preprocessing.promoter_col` to that column's name. This example has no promoter annotation, so promoters are not specially retained; they can still be selected as variable peaks.


In [14]:
data_atac = keep_promoters_and_select_hv_peaks(
    data_atac=data_atac,
    total_n_peaks=ppConfig.num_peaks,
    cluster_key=ppConfig.peak_std_batch_key,
    user_peaks=ppConfig.peaks_user,
    keep_promoters=ppConfig.keep_promoter_peaks,
    promoter_col=ppConfig.promoter_col,  # column in data_atac.var
)

logger.info(f"Final shape after combining promoters + HV => {data_atac.shape}")

request_report = {"rna": data_rna.uns["scdori_requested_features"],
                  "peaks": data_atac.uns["scdori_requested_peaks"]}
(out_dir / "requested_features.json").write_text(json.dumps(request_report, indent=2))
print(json.dumps(request_report, indent=2))

WARNING	Column promoter_col not found in data_atac.var; no special promoter logic.


INFO	Selected top 90000 variable peaks (by std across leiden).


{
  "rna": {
    "genes_retained": [],
    "tfs_retained": [],
    "genes_missing_after_filtering": [],
    "tfs_missing_after_filtering": [],
    "tfs_without_motifs": []
  },
  "peaks": {
    "requested": [],
    "missing": [],
    "outside_gene_windows": [],
    "retained": []
  }
}


## 12. Save the processed RNA and ATAC data

These files contain the selected cells, genes and peaks. Next we prepare the TF-binding and peak–gene information needed for training.


In [15]:
# CSR stores counts by cell row, making training minibatches faster.
# Values and cell/feature order are unchanged. Conversion uses temporary CPU RAM.
# Set preprocessing.count_matrix_csr=false in the shared JSON config to preserve layout.
save_processed_datasets(data_rna, data_atac, out_dir, count_matrix_csr=ppConfig.count_matrix_csr)

INFO	Converting count matrix (56861, 4000) from CSC to CSR for row access (272.6 MiB sparse buffers; temporary additional host memory required).


INFO	Count matrix converted to CSR in 0.180 seconds


INFO	Converting count matrix (56861, 90000) from CSC to CSR for row access (3030.9 MiB sparse buffers; temporary additional host memory required).


INFO	Count matrix converted to CSR in 2.060 seconds


... storing 'gene_type' as categorical


... storing 'chr' as categorical


INFO	Saved processed RNA to data/generated/rna_processed.h5ad with shape=(56861, 4000)


INFO	Saved processed ATAC to data/generated/atac_processed.h5ad with shape=(56861, 90000)


## 13. Save coordinates for the selected peaks

In [16]:
data_atac.var["chr"] = [v.split(":")[0] for v in data_atac.var_names]
data_atac.var["start"] = [
    int(v.split(":")[1].split("-")[0]) for v in data_atac.var_names
]
data_atac.var["end"] = [int(v.split(":")[1].split("-")[1]) for v in data_atac.var_names]
data_atac.var["peak_name"] = data_atac.var_names
peaks_bed = out_dir / "peaks_selected.bed"
data_atac.var[["chr", "start", "end", "peak_name"]].to_csv(
    peaks_bed, sep="\t", header=False, index=False
)

## 14. Find TF motifs in the peaks

We scan the DNA sequences for TF motifs using [FIMO](https://doi.org/10.1093/bioinformatics/btr064) through [tangermeme](https://doi.org/10.1038/s41592-026-03254-z). The motif scores indicate possible TF-binding sites. In the next step, we combine these with expression and accessibility to estimate binding in this dataset.


In [17]:
motif_path = (
    Path(ppConfig.motif_directory)
    / f"{ppConfig.motif_database}_{ppConfig.species}.meme"
)
pwms_sub, key_to_tf = load_motif_database(motif_path, final_tfs)
fasta_path = genome_dir / f"{ppConfig.genome_assembly}.fa"
df_motif_scores = compute_motif_scores(
    bed_file=peaks_bed,
    fasta_file=fasta_path,
    pwms_sub=pwms_sub,
    key_to_tf=key_to_tf,
    n_peaks=data_atac.shape[1],
    window=500,
    threshold=ppConfig.motif_match_pvalue_threshold,
)
df_motif_scores = df_motif_scores[final_tfs]

INFO	Reading motif file: assets/motif_database/cisbp_mouse.meme
INFO	Subselected 300 motifs for 300 TFs.
INFO	Computing motif scores for data/generated/peaks_selected.bed (n_peaks=90000) with window=500
100%|██████████| 300/300 [00:05<00:00, 58.89it/s]

INFO	Finished computing motif scores: (90000, 300)


In [18]:
df_motif_scores.to_csv(out_dir / "motif_scores.tsv", sep="\t")
pd.Series(final_tfs, name="tf").to_csv(out_dir / "tf_order.csv", index=False)


## 15. Estimate TF binding from motifs and the data

Across metacells, we compare each TF's expression with peak accessibility. We keep correlations that stand out from the background of peaks without a motif match, then multiply them by the motif scores.

This produces the predicted binding matrices, called *in-silico ChIP-seq*, used by scDORI. The approach is adapted from [Argelaguet et al.](https://www.biorxiv.org/content/10.1101/2022.06.15.496239v1) and [diffTF](https://pubmed.ncbi.nlm.nih.gov/31801079/).


In [19]:
# TF expression and accessibility of the selected peaks across metacells.
# By default, RNA is the mean of each cell's log-normalized counts, and ATAC the mean fragment count.
rna_matrix, atac_matrix = metacell_correlation_matrices(
    rna_metacell, atac_metacell, data_atac.var_names,
    normalization=ppConfig.correlation_normalization,
)

insilico_chipseq_act, insilico_chipseq_rep = compute_in_silico_chipseq(
    atac_matrix=atac_matrix,
    rna_matrix=rna_matrix,
    motif_scores=df_motif_scores,
    percentile=ppConfig.correlation_percentile,
    n_bg=ppConfig.n_bg_peaks_for_corr,
)
np.save(out_dir / "insilico_chipseq_act.npy", insilico_chipseq_act)
np.save(out_dir / "insilico_chipseq_rep.npy", insilico_chipseq_rep)

INFO	Computing in-silico ChIP-seq correlation...
Thresholding correlation: 100%|██████████| 300/300 [00:01<00:00, 213.67it/s]

INFO	Finished in-silico ChIP-seq computation.


## 16. Compute distance matrix between peaks and genes

This table gives each gene’s distance to each peak. Pairs on different chromosomes are marked `-1`; with the default distance settings, they receive zero weight in the next step.

In [20]:
# distance is set to 0 if the peak midpoint is within gene-body or promoter (5kb upstream of TSS by default)
# distance is -1 if peak-gene pairs on different chromosomes

data_atac.var["index_int"] = range(data_atac.shape[1])
selected_peak_indices = data_atac.var["index_int"].values

# Subset GTF to final genes
gene_info = gtf_df[gtf_df.feature == "gene"].drop_duplicates("gene_name")
gene_info["gene"] = gene_info["gene_name"].values
gene_info = gene_info.set_index("gene_name")
gene_info = gene_info.loc[data_rna.var_names.intersection(gene_info.index)]

gene_info["chr"] = gene_info["seqname"]  # rename col for consistency
# Create gene_coordinates_intersect with necessary columns
gene_info = gene_info[["chr", "start", "end", "strand", "gene"]].copy()
gene_info.index.name = None
gene_info.columns = ["chr_gene", "start", "end", "strand", "gene"]

dist_matrix = compute_gene_peak_distance_matrix(
    data_rna=data_rna, data_atac=data_atac, gene_coordinates_intersect=gene_info
)
np.save(out_dir / "gene_peak_distance_raw.npy", dist_matrix)

INFO	Starting computation of gene-peak distances...
INFO	Number of genes: 4000, Number of peaks: 90000
100%|██████████| 4000/4000 [03:07<00:00, 21.31it/s]

INFO	Gene-peak distance matrix computed with shape: (4000, 90000)


In [21]:
dist_matrix  # -1 denotes peaks on different chromosome

array([[-1, -1, -1, ..., -1, -1, -1],
       [-1, -1, -1, ..., -1, -1, -1],
       [-1, -1, -1, ..., -1, -1, -1],
       ...,
       [-1, -1, -1, ..., -1, -1, -1],
       [-1, -1, -1, ..., -1, -1, -1],
       [-1, -1, -1, ..., -1, -1, -1]])

## 17. Prepare the initial peak–gene weights

Nearby peaks receive larger starting weights. scDORI will adjust these weights during training to describe the expression and accessibility in your data.


In [22]:
dist_matrix[dist_matrix < 0] = 1e8
dist_matrix = np.exp(
    -1 * dist_matrix.astype(float) / ppConfig.peak_distance_scaling_factor
)
dist_matrix = np.where(dist_matrix < ppConfig.peak_distance_min_cutoff, 0, dist_matrix)
np.save(out_dir / "gene_peak_distance_exp.npy", dist_matrix)

## 18. Ready for training

The processed counts, TF-binding matrices and peak–gene information are saved together in `out_dir`.

In the completed gastrulation example shown here:

| | Input | Prepared for training |
| --- | ---: | ---: |
| RNA cells | 65,420 | 56,861 |
| ATAC cells | 57,536 | 56,861 |
| RNA features | 32,285 | 4,000, including 300 TFs |
| ATAC peaks | 192,251 | 90,000 |

For your data, check the final dimensions and requested-feature report above. In particular, check that TFs you want to study were included: adding them later requires repeating preprocessing and training.

Keep this `generated` folder with your fit. Repeating preprocessing can select different peaks, and a phase-1 fit can only be used with the peaks it was trained on (see the [settings guide](../notebook_settings.md)).

Open [training.ipynb](training.ipynb) with the same config. Changing which genes or TFs you *plot* later does not require another fit.


In [23]:
logger.info("=== Pipeline completed successfully ===")
save_notebook_config(settings, "preprocessing", out_dir)
